# Session 09 – Working with Date & Time Data

This notebook generates 3,000 fictional UPI-style transactions and demonstrates datetime parsing, datetime features, time slots, resampling, and UTC-to-India timezone conversion.


In [ ]:
import numpy as np
import pandas as pd

# Task 1: Generate 3,000 fictional UPI transactions (seed 42)
np.random.seed(42)
rng = np.random.default_rng(42)

n = 3000
start = pd.Timestamp("2025-01-01 00:00:00")
end = pd.Timestamp("2025-06-30 23:59:00")
minute_span = int((end - start).total_seconds() // 60)

random_minutes = rng.integers(0, minute_span + 1, size=n)
txn_times = start + pd.to_timedelta(random_minutes, unit="m")
txn_times = pd.Series(txn_times).sort_values(ignore_index=True)

categories = ["Grocery", "Fuel", "Recharge", "Food", "Bills"]
category_probs = [0.28, 0.16, 0.14, 0.27, 0.15]

upi = pd.DataFrame({
    "txn_id": [f"UPI{i:05d}" for i in range(1, n + 1)],
    "txn_time": txn_times.dt.strftime("%d/%m/%Y %H:%M"),
    "amount_inr": rng.integers(10, 5001, size=n),
    "category": rng.choice(categories, size=n, p=category_probs)
})

upi.to_csv("upi_raw.csv", index=False)
print("Saved upi_raw.csv:", upi.shape)
upi.head()


In [ ]:
# Reload raw CSV and parse txn_time using the required format
upi = pd.read_csv("upi_raw.csv")
upi["txn_time"] = pd.to_datetime(
    upi["txn_time"], format="%d/%m/%Y %H:%M"
)

print(upi.dtypes)


## Task 2 – Extract datetime features and create time slots

In [ ]:
upi["year"] = upi["txn_time"].dt.year
upi["month"] = upi["txn_time"].dt.month
upi["day"] = upi["txn_time"].dt.day
upi["weekday"] = upi["txn_time"].dt.day_name()
upi["quarter"] = upi["txn_time"].dt.quarter
upi["hour"] = upi["txn_time"].dt.hour

upi["is_weekend"] = upi["txn_time"].dt.dayofweek >= 5
upi["is_month_end"] = upi["txn_time"].dt.is_month_end

# Use .loc masks, not pd.cut
upi["time_slot"] = "Night"
upi.loc[upi["hour"].between(5, 11), "time_slot"] = "Morning"
upi.loc[upi["hour"].between(12, 16), "time_slot"] = "Afternoon"
upi.loc[upi["hour"].between(17, 21), "time_slot"] = "Evening"

print(upi["time_slot"].value_counts())
upi.head()


## Task 3 – Set datetime index and resample

In [ ]:
upi_indexed = upi.set_index("txn_time").sort_index()

daily_total = upi_indexed["amount_inr"].resample("D").sum()
weekly_total = upi_indexed["amount_inr"].resample("W").sum()
monthly_total = upi_indexed["amount_inr"].resample("ME").sum()

print("Daily grand total:", daily_total.sum())
print("Weekly grand total:", weekly_total.sum())
print("Monthly grand total:", monthly_total.sum())

assert daily_total.sum() == weekly_total.sum() == monthly_total.sum()

daily_total.rename("amount_inr").to_csv("upi_daily.csv")
weekly_total.rename("amount_inr").to_csv("upi_weekly.csv")
monthly_total.rename("amount_inr").to_csv("upi_monthly.csv")


## Task 4 – Convert UTC server time to Asia/Kolkata

In [ ]:
# Treat the original naive timestamps as UTC server time
upi["txn_time_utc"] = upi["txn_time"].dt.tz_localize("UTC")
upi["txn_time_ist"] = upi["txn_time_utc"].dt.tz_convert("Asia/Kolkata")

print("First row before conversion (UTC):", upi["txn_time_utc"].iloc[0])
print("First row after conversion (IST):", upi["txn_time_ist"].iloc[0])

# Compare busiest hour before and after timezone conversion
busiest_utc = upi["txn_time_utc"].dt.hour.value_counts().idxmax()
busiest_ist = upi["txn_time_ist"].dt.hour.value_counts().idxmax()
print("Busiest hour before conversion (UTC):", busiest_utc)
print("Busiest hour after conversion (IST):", busiest_ist)

# Save final clean dataset
upi["txn_time"] = upi["txn_time"].dt.strftime("%Y-%m-%d %H:%M:%S")
upi["txn_time_utc"] = upi["txn_time_utc"].astype(str)
upi["txn_time_ist"] = upi["txn_time_ist"].astype(str)
upi.to_csv("upi_clean.csv", index=False)
print("Saved upi_clean.csv")


## Final verification
- 3,000 transactions generated.
- All timestamps parse using `%d/%m/%Y %H:%M`.
- Time slots are assigned with `.loc` masks.
- Daily, weekly, and monthly resampling totals are equal.
- UTC to Asia/Kolkata conversion shifts timestamps by 5 hours 30 minutes.
- Final output: `upi_clean.csv`.
